# Alpha-FX: Intelligent Forex Trading System

Welcome to **Alpha-FX**, a Deep Reinforcement Learning (DRL) system designed to trade foreign exchange markets.

## 🎯 Goal
The goal of this notebook is to walk you through the entire lifecycle of an algorithmic trading agent:
1.  **DataOps**: Ingesting and processing financial data (Prices, Indicators, Macro yields).
2.  **Training**: Teaching an AI agent (PPO) to trade profitably.
3.  **Evaluation**: Testing the agent against a "Buy & Hold" baseline to verify performance.

## 🌍 FX Context
We trade a portfolio of major currency pairs (`EURUSD`, `GBPUSD`, etc.) against the US Dollar.
*   **The Strategy**: The agent learns to allocate portfolio weights (e.g., 50% EUR, 50% USD) to maximize returns while managing risk (volatility).
*   **The Edge**: We provide the agent with **Macroeconomic Data** (US Treasury Yields) so it can understand interest rate trends (Carry Trade).

---

### 🔎 System Overview
The data flows from raw Yahoo Finance inputs to a trained model.

```mermaid
graph LR
    A[Yahoo Finance] -->|Fetch| B(Data Pipeline)
    B -->|Clean & Feature Eng| C[(Parquet Dataset)]
    C -->|Train| D[RL Agent (PPO)]
    D -->|Evaluate| E[Backtest & Benchmark]
```

## 1. Setup & Dependencies
First, we ensure all necessary libraries are installed and importable.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
import os
import pandas as pd
import numpy as np
import gymnasium as gym
import stable_baselines3
import matplotlib.pyplot as plt
import torch

# Add root to path so we can import local modules
sys.path.append(os.path.abspath('.'))

print(f"Pandas: {pd.__version__}")
print(f"Gymnasium: {gym.__version__}")
print(f"Stable Baselines3: {stable_baselines3.__version__}")

# Check for GPU
device = 'cpu'
if torch.cuda.is_available():
    device = 'cuda'
    print("🚀 NVIDIA CUDA GPU Detected!")
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    device = 'mps'
    print("🍎 Apple Metal GPU (MPS) Detected!")
else:
    print("💻 Running on CPU.")

## 2. Data Operations (DataOps)

We need to fetch historical data for our currency pairs. We also fetch **Macro Data** (US 10-Year Treasury Yield `^TNX`) to help the agent understand the broader economic environment.

### Pipeline Steps:
1.  **Fetch**: Download OHLCV data from Yahoo Finance (2018-2023).
2.  **Clean**: Fix missing values (Market holidays).
3.  **Feature Engineering**: Add technical indicators:
    *   **Trend**: MACD, ADX.
    *   **Volatility**: ATR, Bollinger Bands.
    *   **Momentum**: RSI, Williams %R.
    *   **Macro**: US Risk-Free Rate (Yield).
4.  **Save**: Output to `data/fx_data_2018_2023.parquet`.

In [ ]:
from planning.run_pipeline import main as run_pipeline

# Customize Data Scope
TICKERS = ['EURUSD=X', 'GBPUSD=X', 'JPY=X', 'SEK=X', 'EURSEK=X']
START_DATE = '2018-01-01'
END_DATE = '2023-01-01'

# Run the Data Pipeline
run_pipeline(tickers=TICKERS, start_date=START_DATE, end_date=END_DATE)

In [ ]:
# Inspect the generated data
df = pd.read_parquet('data/fx_data_2018_2023.parquet')
print("Dataset Shape:", df.shape)
print("Columns:", df.columns.tolist())
print("\nSample Data:")
df.head()

## 3. Training the Agent

We use **PPO (Proximal Policy Optimization)**, a state-of-the-art reinforcement learning algorithm. 

You can now **Fine-Tune** the agent strategies using these parameters:
1.  **`total_timesteps`**: How long the agent practices. (Default: 30,000. Try 100k+ for real results).
2.  **`lookback`**: The window of historical data the agent sees. (Default: 10 days. Try 30 to capture monthly trends).
3.  **`ent_coef`**: Exploration rate. Higher values (e.g., 0.01) encourage trying new things; lower values stick to what works.
4.  **`device`**: 'auto', 'cuda' (Nvidia), 'mps' (Mac), or 'cpu'. Using GPU significantly speeds up training.

In [ ]:
from train_agent import train

# Interactive Training Control
# Advanced: Increase lookback to 30 days and add some exploration
# Using the detected device from Step 1
train(
    total_timesteps=30000, 
    lookback=30,           # Window size
    ent_coef=0.01,         # encourage exploration
    device=device          # Use GPU if available
)

### 📊 Training Visualization
Let's see how the agent learned over time. We plot the **Episode Reward** (smoothed).
*   **Upward Trend**: The agent is learning strategies that generate more return (or lose less).
*   **Flat/Down**: The agent is struggling or market conditions are too chaotic.

In [ ]:
# Plot Learning Curve
try:
    # SB3 Monitor logs file format: first line is metadata, second line is header
    log_path = 'results/monitor.csv'
    if os.path.exists(log_path):
        log_df = pd.read_csv(log_path, skiprows=1)
        # Format: r (rewards), l (length), t (time)
        if 'r' in log_df.columns:
            # Smooth it for readability
            log_df['smoothed_reward'] = log_df['r'].rolling(window=10).mean()
            
            plt.figure(figsize=(10, 6))
            plt.plot(log_df.index, log_df['smoothed_reward'], label='Smoothed Reward', color='blue')
            plt.plot(log_df.index, log_df['r'], label='Raw Reward', color='lightblue', alpha=0.3)
            plt.title(f"Agent Learning Curve ({len(log_df)} episodes)")
            plt.xlabel("Episodes")
            plt.ylabel("Episode Reward")
            plt.legend()
            plt.grid(True, alpha=0.3)
            plt.show()
        else:
            print("Log file found but missing 'r' column.")
    else:
        print(f"No training logs found at {log_path}. Run training first.")
except Exception as e:
    print(f"Could not plot learning curve: {e}")

## 4. Evaluation & Benchmarking

Does the agent actually make money? We compare it against a **Benchmark**: 
*   **Strategy**: Buy & Hold (Equal Weights).
*   **Dataset**: 2022 Validation Set (Unseen during training).

We look at:
1.  **Cumulative Return**: Total profit %.
2.  **Sharpe Ratio**: Risk-adjusted return (Higher is better).
3.  **Max Drawdown**: Maximum peak-to-trough drop (Risk).

In [ ]:
from benchmark_agent import run_benchmark

# Run Benchmark Analysis
# IMPORTANT: Must match 'lookback' used in training (30)
run_benchmark(lookback=30)

### 📈 Performance Visualziation (Equity Curve)
Visual comparison of the Agent's portfolio value vs the Benchmark over time.

In [ ]:
# Plot Equity Curve
try:
    eq_path = 'results/equity.csv'
    if os.path.exists(eq_path):
        eq_df = pd.read_csv(eq_path)
        if 'date' in eq_df.columns:
            eq_df['date'] = pd.to_datetime(eq_df['date'])
            x_axis = eq_df['date']
        else:
            x_axis = eq_df.index
            
        plt.figure(figsize=(12, 6))
        plt.plot(x_axis, eq_df['agent'], label='Alpha-FX Agent', color='green', linewidth=2)
        plt.plot(x_axis, eq_df['baseline'], label='Benchmark (B&H)', color='gray', linestyle='--')
        
        plt.title("Performance Comparison: Agent vs Benchmark")
        plt.xlabel("Date")
        plt.ylabel("Portfolio Value ($)")
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.show()
    else:
        print(f"No equity data found at {eq_path}. Run benchmark first.")
except Exception as e:
    print(f"Could not plot equity curve: {e}")

## 5. Conclusion

You have successfully ran the Alpha-FX pipeline.

*   **Next Steps**: 
    *   Try increasing `TOTAL_TIMESTEPS` to `100000` to see if the agent learns a better policy.
    *   Add more tickers (e.g., `'AUDUSD=X'`) in the DataOps section.